## DuckDB vs Spark on ADLFS G2(ABFS) setup and speed test on single node VM

Instead of using expensive PaaS such as Azure Synapse and Databricks, equivalent Spark workload can be performed on the cheaper, more flexible and portable VMs or K8s.

This notebook demonstrates how to set up VM environment for high performance DuckDB and Spark queries on Azure Data Lake, and performs a small scale speed test

DuckDB wins the test (DuckDB 101s vs Spark 175s on Azure D16plds_v6 VM)


#### Environment Setup
- Azure VM have blob storage contributor to ABFS via managed identity(environment variable AZCOPY_MSI_CLIENT_ID)
- get hadoop-client version from Spark or pySpark jar directory, e.g. 3.5.0
- VM need network connection to maven and duckdb repositories to download libraries for ABFS access
- VM temporary SSD is mounted under /mnt with proper ownership and permissions
- ADLS container mounted by blobfuse2 for DuckDB access (DuckDB azure extension has issues accessing storage account)

#### Set up DuckDB connection

In [1]:
import duckdb, os

In [2]:
client_id = os.environ.get('AZCOPY_MSI_CLIENT_ID')
if not client_id:
    raise RuntimeError('Environment variable AZCOPY_MSI_CLIENT_ID is not set')

# Ensure azure extension is available and loaded
duckdb.sql('INSTALL azure;LOAD azure;')
duckdb.sql("SET temp_directory = '/mnt/'")

In [ ]:
duckdb.sql(f"CREATE SECRET adls_mi (TYPE AZURE, PROVIDER MANAGED_IDENTITY, ACCOUNT_NAME '[REDACTED]', CLIENT_ID '{client_id}');")

┌─────────┐
│ Success │
│ boolean │
├─────────┤
│ true    │
└─────────┘

#### Test DuckDB connection to ADLS

In [ ]:
duckdb.sql("CREATE TEMPORARY VIEW nyctaxi AS SELECT * FROM read_parquet('az://[REDACTED].blob.core.windows.net/src/yellow/**/*.parquet')")

In [ ]:
duckdb.sql("SELECT * FROM nyctaxi LIMIT 10").show()

┌──────────┬─────────────────────┬─────────────────────┬────────────────┬──────────────┬──────────────┬──────────────┬──────────┬──────────┬────────┬────────┬────────────┬─────────────────┬─────────────┬────────────┬────────┬────────┬──────────────────────┬───────────┬─────────────┬─────────────┬─────────┬────────┐
│ vendorID │ tpepPickupDateTime  │ tpepDropoffDateTime │ passengerCount │ tripDistance │ puLocationId │ doLocationId │ startLon │ startLat │ endLon │ endLat │ rateCodeId │ storeAndFwdFlag │ paymentType │ fareAmount │ extra  │ mtaTax │ improvementSurcharge │ tipAmount │ tollsAmount │ totalAmount │ puMonth │ puYear │
│ varchar  │      timestamp      │      timestamp      │     int32      │    double    │   varchar    │   varchar    │  double  │  double  │ double │ double │   int32    │     varchar     │   varchar   │   double   │ double │ double │       varchar        │  double   │   double    │   double    │  int64  │ int64  │
├──────────┼─────────────────────┼───────────────

#### DuckDB Speedtest

In [ ]:
duckdb.sql(r"""
SELECT 
    puLocationId,
    puYear,
    puMonth,
    EXTRACT(HOUR FROM tpepPickupDateTime) AS pickup_hour,
    COUNT(*) AS total_trips,
    SUM(passengerCount) AS total_passengers,
    AVG(tripDistance) AS avg_trip_distance,
    AVG(DATE_DIFF('second', tpepPickupDateTime, tpepDropoffDateTime) / 60.0) AS avg_trip_duration_minutes,
    SUM(fareAmount) AS total_fare_revenue,
    SUM(tipAmount) AS total_tip_revenue
FROM 
    read_parquet('/adls/yellow/**/*.parquet') AS nyctaxi
WHERE 
    tripDistance > 0 
    AND fareAmount > 0
GROUP BY 
    puLocationId,
    puYear,
    puMonth,
    EXTRACT(HOUR FROM tpepPickupDateTime)
ORDER BY 
    total_fare_revenue DESC;
""").write_parquet('/mnt/duckdb_nyctaxi_agg.parquet', compression='snappy')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

#### Create spark session with ADLFS G2 support

In [ ]:
import os
from pyspark.sql import SparkSession

In [ ]:
# Same managed identity as DuckDB
client_id = os.environ.get('AZCOPY_MSI_CLIENT_ID')
if not client_id:
    raise RuntimeError('Environment variable AZCOPY_MSI_CLIENT_ID is not set')

hadoop_version = '3.5.0' # from pySpark installation jar directory

account_name = "[REDACTED]"  # from your ABFS path: abfss://src@[REDACTED].dfs.core.windows.net/

spark = SparkSession.builder \
    .appName("SingleTaskApp") \
    .master("local[*]") \
    .config("spark.local.dir", "/mnt") \
    .config("spark.jars.packages",
            f"org.apache.hadoop:hadoop-common:{hadoop_version},org.apache.hadoop:hadoop-azure:{hadoop_version},com.azure:azure-storage-blob:12.34.0") \
    .config(f"fs.azure.account.auth.type.{account_name}.dfs.core.windows.net", "OAuth") \
    .config(f"fs.azure.account.oauth.provider.type.{account_name}.dfs.core.windows.net",
            "org.apache.hadoop.fs.azurebfs.oauth2.MsiTokenProvider") \
    .config(f"fs.azure.account.oauth2.client.id.{account_name}.dfs.core.windows.net", client_id) \
    .getOrCreate()

sc = spark.sparkContext

:: loading settings :: url = jar:file:/home/[REDACTED]/.venv/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/[REDACTED]/.ivy2.5.2/cache
The jars for the packages stored in: /home/[REDACTED]/.ivy2.5.2/jars
org.apache.hadoop#hadoop-common added as a dependency
org.apache.hadoop#hadoop-azure added as a dependency
com.azure#azure-storage-blob added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-6b337359-3a2b-4346-ae30-e3c249c87ef2;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-common;3.5.0 in central
	found org.apache.hadoop.thirdparty#hadoop-shaded-protobuf_3_25;1.5.0 in central
	found org.apache.hadoop#hadoop-annotations;3.5.0 in central
	found org.apache.hadoop.thirdparty#hadoop-shaded-guava;1.5.0 in central
	found com.google.guava#guava;33.4.8-jre in central
	found com.google.guava#failureaccess;1.0.3 in central
	found com.google.guava#listenablefuture;9999.0-

#### Test spark connection to ADLS

In [5]:
spark.read.parquet('abfss://src@[REDACTED].dfs.core.windows.net/yellow').limit(10).show()

+--------+-------------------+-------------------+--------------+------------+------------+------------+----------+---------+----------+---------+----------+---------------+-----------+----------+-----+------+--------------------+---------+-----------+-----------+------+-------+
|vendorID| tpepPickupDateTime|tpepDropoffDateTime|passengerCount|tripDistance|puLocationId|doLocationId|  startLon| startLat|    endLon|   endLat|rateCodeId|storeAndFwdFlag|paymentType|fareAmount|extra|mtaTax|improvementSurcharge|tipAmount|tollsAmount|totalAmount|puYear|puMonth|
+--------+-------------------+-------------------+--------------+------------+------------+------------+----------+---------+----------+---------+----------+---------------+-----------+----------+-----+------+--------------------+---------+-----------+-----------+------+-------+
|     CMT|2012-02-29 23:53:14|2012-03-01 00:00:43|             1|         2.1|        NULL|        NULL|-73.980494|40.730601|-73.983532|40.752311|         1|   

#### Spark Speedtest

In [6]:
spark.read.parquet('abfss://src@[REDACTED].dfs.core.windows.net/yellow').createOrReplaceTempView('nyctaxi')

In [7]:
spark.sql(r"""
SELECT 
    puLocationId,
    puYear,
    puMonth,
    HOUR(tpepPickupDateTime) AS pickup_hour,
    COUNT(1) AS total_trips,
    SUM(passengerCount) AS total_passengers,
    AVG(tripDistance) AS avg_trip_distance,
    AVG((UNIX_TIMESTAMP(tpepDropoffDateTime) - UNIX_TIMESTAMP(tpepPickupDateTime)) / 60.0) AS avg_trip_duration_minutes,
    SUM(fareAmount) AS total_fare_revenue,
    SUM(tipAmount) AS total_tip_revenue
FROM 
    nyctaxi
WHERE 
    tripDistance > 0 
    AND fareAmount > 0
GROUP BY 
    puLocationId,
    puYear,
    puMonth,
    HOUR(tpepPickupDateTime)
ORDER BY 
    total_fare_revenue DESC;
""").write.parquet('/mnt/spark_nyctaxi_agg.parquet', compression='snappy')

26/05/28 06:42:35 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/05/28 06:42:35 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/05/28 06:42:35 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/05/28 06:42:35 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/05/28 06:42:35 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
